# Daily Device Detections Exploration

This notebook explores daily detection patterns across AudioMoth devices and sites. Detection counts are summarised by hour of day at the overall, site and device level to examine diel activity patterns and temporal variation in acoustic activity.

## This Notebook Covers:

- Total detections by hour of day (all devices combined)

- Detections by hour of the day for the site as whole, and per device

- Comparison of diel patterns across devices and sites

## Setup System Path and Get Data

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd


# Go up one level to .../audiomoth
PROJECT_ROOT = Path(os.getcwd()).resolve().parent

# Add project root to sys.path so `src` is importable
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DATA_PATH = Path(PROJECT_ROOT) / "data_processed" / "analysis_df.parquet"
analysis_df = pd.read_parquet(PROCESSED_DATA_PATH)

# Make pandas show more columns/rows while exploring
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## Overall

In [ ]:
detections_by_hour = (
    analysis_df.groupby("hour")
    .size()
    .reset_index(name="detections")
    .sort_values("hour")
)

detections_by_hour

### Save

In [ ]:
import src.data_store as data_store

data_store.save_dataframe_to_csv(
    detections_by_hour,
    Path(PROJECT_ROOT) / "outputs",
    "overall_detections_hourly_summary",
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(detections_by_hour["hour"], detections_by_hour["detections"])
plt.xlabel("Hour of Day")
plt.ylabel("Total Detections")
plt.title("Total Detections by Hour (All Devices Combined)")
plt.show()

Detection counts show a pronounced dawn chorus peak between 05:00 and 08:00, with activity peaking at 07:00. Detections gradually decline throughout the day, with a minor increase during early evening, before dropping to low nocturnal levels. This pattern is consistent with expected behaviour.

## Detections by Hour of Day Per Site

### Goal:

For each site and hour:

Total detections

Site detections per recording hour (effort standardised)

Per-device detections per recording hour

Device % contribution within site per hour

### Assumption:
Active day = 24 recording hours

Therefore each active device day contributes 1 recording hour to each clock hour bin.

In [ ]:
# Get the number of unique days each device was active (had detections)
# to get active device days per device.
device_effort = (
    analysis_df.groupby(["site", "device"])["date"]
    .nunique()
    .reset_index(name="active_device_days")
)

In [ ]:
# Get the number of detections per site, device, and hour to get device-hourly detection counts.
device_hourly = (
    analysis_df.groupby(["site", "device", "hour"])
    .size()
    .reset_index(name="detections")
)

In [ ]:
# Merge device-hourly detections with device effort.
device_hourly = device_hourly.merge(device_effort, on=["site", "device"])

# Calculate detections per recording hour
device_hourly["detections_per_recording_hour"] = (
    device_hourly["detections"] / device_hourly["active_device_days"]
)

device_hourly

In [ ]:
""" Here we are taking the device-hourly detections and summing them up to get 
site-hourly detections, and also summing the active device days to 
get site-hourly effort. """

site_hourly = (
    device_hourly.groupby(["site", "hour"])
    .agg({"detections": "sum", "active_device_days": "sum"})
    .reset_index()
)

# Now we can calculate detections per recording hour at the site level.

site_hourly["site_detections_per_recording_hour"] = (
    site_hourly["detections"] / site_hourly["active_device_days"]
)

site_hourly

### Save

In [ ]:
import src.data_store as data_store

data_store.save_dataframe_to_csv(
    site_hourly,
    Path(PROJECT_ROOT) / "outputs",
    "site_detections_hourly_summary",
)

In [ ]:
# Find total detections per site at each hour
device_hourly = device_hourly.merge(
    site_hourly[["site", "hour", "detections"]],
    on=["site", "hour"],
    suffixes=("", "_site_total"),
)

# Find the percentage of detections each device contributes to the site total
# at each hour
device_hourly["percent_within_site_hour"] = (
    device_hourly["detections"] / device_hourly["detections_site_total"] * 100
)

device_hourly

### Save

In [ ]:
import src.data_store as data_store

data_store.save_dataframe_to_csv(
    device_hourly,
    Path(PROJECT_ROOT) / "outputs",
    "device_detections_daily_summary",
)

## Quietest 3 Hour Period For Each Site

In [ ]:
# Assuming a window of reasonable working hours between 7am-6pm.
working_hours = site_hourly[site_hourly["hour"].between(7, 18)]

# Find the 3 quietest hours (lowest detections per recording hour) for each site
quietest_hours = (
    working_hours.sort_values(["site", "site_detections_per_recording_hour"])
    .groupby("site")
    .head(3)
    .sort_values(["site", "hour"])
)

quietest_hours

In [ ]:
quietest_hours["hour_label"] = (
    quietest_hours["hour"].astype(str)
    + ":00–"
    + (quietest_hours["hour"] + 1).astype(str)
    + ":00"
)
quietest_hours[["site", "hour_label", "site_detections_per_recording_hour"]]

Across sites, the three lowest-intensity working hours (07:00–18:00) consistently fall within the early–mid afternoon window, typically between 14:00 and 17:00. This pattern is consistent across multiple sites and reflects the natural decline in vocal activity following the morning peak.

While absolute detection rates vary between sites, the relative temporal pattern is broadly similar. Early–mid afternoon therefore represents the most suitable period for undertaking reserve management activities where minimising acoustic disturbance is a priority.

## Detections within Diel Period 

In [ ]:
def assign_diel(hour):
    if hour >= 20 or hour <= 3:
        return "Night"
    elif 4 <= hour <= 7:
        return "Dawn"
    elif 8 <= hour <= 16:
        return "Day"
    elif 17 <= hour <= 19:
        return "Dusk"


analysis_df["diel_period"] = analysis_df["hour"].apply(assign_diel)

In [ ]:
detections_by_diel = (
    analysis_df.groupby("diel_period").size().reset_index(name="detections")
)

detections_by_diel

In [ ]:
diel_order = ["Night", "Dawn", "Day", "Dusk"]

analysis_df["diel_period"] = pd.Categorical(
    analysis_df["diel_period"], categories=diel_order, ordered=True
)

detections_by_diel = (
    analysis_df.groupby("diel_period", observed=True)
    .size()
    .reset_index(name="detections")
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(detections_by_diel["diel_period"], detections_by_diel["detections"])
plt.xlabel("Diel Period")
plt.ylabel("Total Detections")
plt.title("Total Detections by Diel Period")
plt.show()

Detections were highest during daytime hours, followed by dawn. Although dawn exhibited the strongest concentrated hourly peak, the broader daytime window resulted in the greatest overall detection volume. Dusk and night periods contributed comparatively fewer detections, reflecting predominantly diurnal vocal behaviour across the monitored sites.

This summary does not account for variation in active device days or recording effort within each hourly window. Further analysis should standardise detections by active device days and recording hours per diel period to enable more robust comparison.